# FormatBench Baseline Classifier

A DistilBERT baseline trained on the FormatBench preference dataset 
hits 100% test accuracy but only 1.2% on the held-out adversarial set, demonstrating that the dataset's surface signal hides a reward-hacking 
trap. Validates the need for the adversarial evaluation pipeline 
described in the Prosify project.

This notebook is part of the [Prosify project](https://github.com/krishyaid-coder/prosify.git), which is building a DPO-trained model to correct over-formatting bias in LLMs.

## What it does

Trains a small DistilBERT classifier on the [FormatBench dataset](https://www.kaggle.com/datasets/techiekd/formatbench-llm-formatting-bias-dataset) to distinguish appropriate-prose responses from over-formatted ones. Then evaluates the same trained classifier on the dataset's adversarial held-out set, examples where structured responses (recipes, install instructions,comparisons) are the *correct* answer.

## The finding

- **100% accuracy** on the main held-out test set
- **1.2% accuracy** on the adversarial held-out set (worse than random)

The classifier didn't learn "is this response appropriate for the prompt?" 
It learned "does this text contain markdown structure?" and which gets it 
to 100% on the main test set but fails systematically on cases where 
structure is genuinely correct.

This validates two things at once:

1. The FormatBench dataset has clear, learnable preference signal.
2. The signal is largely *surface-level*, meaning naive training on the 
   dataset (DPO included) is at high risk of producing a model that 
   over-generalizes to "always strip structure."

## Why this matters for the eventual DPO model

The 100% / 1.2% gap defines the bar that the project's eventual DPO-trained 
model needs to beat. A successful model needs to score well on **both** 
the main test set and the adversarial set, demonstrating learned 
context-sensitivity rather than surface mimicry.

## Reproducibility

- Random seed: 42 (matches the `build_splits.py` script in the Prosify repo)
- All hyperparameters in the notebook
- Code available at: https://github.com/krishyaid-coder/prosify.git
---

*Part of the [Prosify project](https://github.com/krishyaid-coder/prosify.git). 
Dataset on [Hugging Face](https://huggingface.co/datasets/krishy-d/formatbench/discussions/1#6a1a2b44ace3d692e49cd5b5) and [Kaggle](https://www.kaggle.com/datasets/techiekd/formatbench-llm-formatting-bias-dataset).*

## 1. Install dependencies

On Kaggle, `transformers` and `datasets` are usually pre-installed but may be outdated. We install the latest to be safe.

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn

## 2. Configuration



In [ ]:
# ===== HUGGING FACE =====
HF_USERNAME = "krishy-d"  # <-- your HuggingFace username
# =====================

DATASET_REPO = f"{HF_USERNAME}/formatbench"
MODEL_NAME = "distilbert-base-uncased"  # small, fast, reliable for baseline
MAX_LENGTH = 512
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
SEED = 42  # must match the split seed in build_splits.py

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Load the FormatBench dataset

We load the `default` config (the 551 main training pairs). The adversarial held-out set is in the `adversarial` config and we won't touch it in this notebook — it's reserved for end-of-pipeline evaluation.

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_REPO, split="train")
print(f"Loaded {len(raw)} preference pairs")
print(f"\nFirst row schema: {list(raw[0].keys())}")
print(f"\nExample row:")
print(f"  prompt:   {raw[0]['prompt'][:80]}...")
print(f"  chosen:   {raw[0]['chosen'][:80]}...")
print(f"  rejected: {raw[0]['rejected'][:80]}...")
print(f"  context:  {raw[0]['context']}")

## 4. Recreate the train/val/test splits

We replicate the same stratified splits used in the local `build_splits.py` script — 80/10/10, stratified by `context`, with the same seed. Re-running with the same seed produces identical splits.

In [ ]:
from collections import defaultdict
from datasets import Dataset

rng = random.Random(SEED)
rows = list(raw)

by_context = defaultdict(list)
for row in rows:
    by_context[row["context"]].append(row)

train_rows, val_rows, test_rows = [], [], []
for context, items in sorted(by_context.items()):
    shuffled = items[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, n // 10)
    n_val = max(1, n // 10)
    n_train = n - n_test - n_val
    train_rows.extend(shuffled[:n_train])
    val_rows.extend(shuffled[n_train:n_train + n_val])
    test_rows.extend(shuffled[n_train + n_val:])

rng.shuffle(train_rows)
rng.shuffle(val_rows)
rng.shuffle(test_rows)

print(f"Train: {len(train_rows)}  |  Val: {len(val_rows)}  |  Test: {len(test_rows)}")

## 5. Expand each preference pair into two classification examples

Each row of the form `(prompt, chosen, rejected)` becomes two training examples:
- `(prompt + chosen)` → label `1` (preferred response)
- `(prompt + rejected)` → label `0` (not preferred)

The classifier learns to look at a (prompt, response) pair and predict whether the response is the preferred one. If our dataset has signal, the classifier should be able to do this well above chance (50%).

In [ ]:
def expand_to_classifier_format(rows_list):
    expanded = []
    for row in rows_list:
        # chosen example
        expanded.append({
            "text": f"{row['prompt']}\n\n{row['chosen']}",
            "label": 1,
            "context": row["context"],
        })
        # rejected example
        expanded.append({
            "text": f"{row['prompt']}\n\n{row['rejected']}",
            "label": 0,
            "context": row["context"],
        })
    return expanded

train_cls = Dataset.from_list(expand_to_classifier_format(train_rows))
val_cls = Dataset.from_list(expand_to_classifier_format(val_rows))
test_cls = Dataset.from_list(expand_to_classifier_format(test_rows))

print(f"Train: {len(train_cls)} classifier examples ({len(train_cls) // 2} pairs x 2)")
print(f"Val:   {len(val_cls)}")
print(f"Test:  {len(test_cls)}")

## 6. Tokenize

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dynamic padding via DataCollator
    )

train_tok = train_cls.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_cls.map(tokenize_fn, batched=True, remove_columns=["text"])
test_tok = test_cls.map(tokenize_fn, batched=True, remove_columns=["text"])

print("Tokenization complete")

## 7. Set up the model and trainer

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

training_args = TrainingArguments(
    output_dir="./classifier_out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    seed=SEED,
    save_total_limit=1,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Trainer ready. Training will start with the next cell.")

## 8. Train

On a T4, this should take ~10-15 minutes for 3 epochs.

In [ ]:
trainer.train()

## 9. Evaluate on the held-out test set

The validation set was used for model selection (best checkpoint by val accuracy). The test set has been untouched throughout training — this is the real result.

In [ ]:
test_results = trainer.evaluate(test_tok)
test_accuracy = test_results["eval_accuracy"]

print(f"=== TEST SET RESULTS ===\n")
print(f"  Test accuracy: {test_accuracy:.4f}  ({test_accuracy:.1%})")
print(f"  Test examples: {len(test_tok)}")
print(f"  Random baseline: 50% (binary classification)")

if test_accuracy >= 0.90:
    verdict = "Strong signal — dataset is clearly learnable. Move forward with DPO."
elif test_accuracy >= 0.80:
    verdict = "Decent signal — dataset is learnable but with some noise. Worth examining where the model fails."
elif test_accuracy >= 0.70:
    verdict = "Weak signal — the dataset may be noisier than ideal. Consider reviewing dataset quality before DPO."
else:
    verdict = "Weak signal — likely an issue with the dataset or training setup. Investigate before proceeding."

print(f"\n  Verdict: {verdict}")

## 10. Per-context accuracy

Breaking down accuracy by category tells us where the dataset has the strongest and weakest signal. Categories where the classifier struggles are worth examining manually — they may have noisier preference labels.

In [ ]:
predictions = trainer.predict(test_tok)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

context_results = defaultdict(lambda: [0, 0])  # context -> [correct, total]
for ctx, pred, true in zip(test_cls["context"], pred_labels, true_labels):
    context_results[ctx][1] += 1
    if pred == true:
        context_results[ctx][0] += 1

print(f"=== PER-CONTEXT ACCURACY ===\n")
print(f"  {'context':<22} {'correct':>10}  {'accuracy':>10}")
print(f"  {'-' * 22}   {'-' * 8}    {'-' * 8}")
for ctx in sorted(context_results.keys()):
    correct, total = context_results[ctx]
    acc = correct / total
    print(f"  {ctx:<22} {correct:>4}/{total:<4}  {acc:>9.1%}")

## 11. Sample predictions

Let's look at a few examples to make sure the model is actually doing something sensible, not just memorizing surface patterns.

In [ ]:
import torch.nn.functional as F

model.eval()
device = next(model.parameters()).device

# Show 5 random test examples with predictions and confidence
import random as _random
_random.seed(SEED)
sample_indices = _random.sample(range(len(test_cls)), 5)

for idx in sample_indices:
    example = test_cls[idx]
    text = example["text"]
    true_label = example["label"]
    context = example["context"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0]
        pred_label = int(torch.argmax(probs))
        confidence = float(probs[pred_label])

    label_names = {0: "rejected", 1: "chosen"}
    correct = "✓" if pred_label == true_label else "✗"

    print(f"--- Example (context: {context}) ---")
    print(f"Text: {text[:200]}{'...' if len(text) > 200 else ''}")
    print(f"True: {label_names[true_label]}")
    print(f"Pred: {label_names[pred_label]}  (confidence: {confidence:.1%})  {correct}")
    print()

## 12. Save the model (optional)

If you want to share this baseline classifier, you can push it to the Hugging Face Hub. Uncomment and adjust the cell below.

Saving the classifier isn't strictly necessary for the project and its role is just to validate that the dataset has signal. But pushing it to HF makes the baseline reproducible.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()  # paste your HF token
# trainer.push_to_hub(f"{HF_USERNAME}/formatbench-distilbert-baseline")

print("Cell intentionally not run — uncomment to push the trained classifier to HuggingFace.")

## Summary

If we get test accuracy in the 85-95% range,then the dataset has clear, learnable signal. This validates that proceeding to DPO fine-tuning is worth the effort.

**What this experiment did NOT do** (intentionally):
- This is a classifier, not a generative model. It can tell which response is preferred, but it can't generate prose itself.
- It doesn't use the adversarial held-out set and that's reserved for evaluating the eventual DPO-trained generative model.

**Next step in the Prosify project:** set up DPO fine-tuning of a small base model (Qwen 2.5 1.5B or Llama 3.2 1B) using the `train.jsonl` split, with the same val/test splits used here for selection and evaluation.

In [ ]:
# === Adversarial sanity check ===
# Load the adversarial held-out set and see how the classifier 
# (trained only on the main dataset) handles it.

adversarial = load_dataset(DATASET_REPO, "adversarial", split="test")
print(f"Loaded {len(adversarial)} adversarial examples")

# Expand to classifier format (chosen=1, rejected=0)
adv_cls = Dataset.from_list(expand_to_classifier_format(list(adversarial)))
adv_tok = adv_cls.map(tokenize_fn, batched=True, remove_columns=["text"])

# Predict
adv_predictions = trainer.predict(adv_tok)
adv_pred_labels = np.argmax(adv_predictions.predictions, axis=1)
adv_true_labels = adv_predictions.label_ids
adv_accuracy = (adv_pred_labels == adv_true_labels).mean()

print(f"\n=== ADVERSARIAL SET RESULTS ===\n")
print(f"  Adversarial accuracy: {adv_accuracy:.4f}  ({adv_accuracy:.1%})")
print(f"  (Compare to: 100% on main test set, 50% random baseline)")

# Per-topic breakdown
adv_topic_results = defaultdict(lambda: [0, 0])
for topic, pred, true in zip(adv_cls["context"], adv_pred_labels, adv_true_labels):
    # The 'context' here is 'adversarial_structured' for all rows
    # Use topic_area instead for breakdown if it's in the dataset
    pass

# Use topic_area if available, else context
topic_field = "topic_area" if "topic_area" in adv_cls.column_names else "context"
adv_topic_results = defaultdict(lambda: [0, 0])
for topic, pred, true in zip(adv_cls[topic_field], adv_pred_labels, adv_true_labels):
    adv_topic_results[topic][1] += 1
    if pred == true:
        adv_topic_results[topic][0] += 1

print(f"\n=== PER-TOPIC ACCURACY ON ADVERSARIAL SET ===\n")
print(f"  {'topic':<25} {'correct':>10}  {'accuracy':>10}")
for topic in sorted(adv_topic_results):
    c, t = adv_topic_results[topic]
    print(f"  {topic:<25} {c:>4}/{t:<4}  {c/t:>9.1%}")

# Also look at confidence
import torch.nn.functional as F
adv_probs = F.softmax(torch.tensor(adv_predictions.predictions), dim=-1)
chosen_examples = [i for i, ex in enumerate(adv_cls) if ex["label"] == 1]
rejected_examples = [i for i, ex in enumerate(adv_cls) if ex["label"] == 0]

mean_prob_for_chosen = adv_probs[chosen_examples, 1].mean()
mean_prob_for_rejected = adv_probs[rejected_examples, 0].mean()
print(f"\nMean confidence on adversarial 'chosen' (structured): {mean_prob_for_chosen:.1%}")
print(f"Mean confidence on adversarial 'rejected' (prose): {mean_prob_for_rejected:.1%}")